**GROUP BY ON DATA SET**

In [ ]:
import sqlite3
import pandas as pd

df = pd.read_csv("train.csv")
conn = sqlite3.connect("housing.db")
df.to_sql("houses", conn, if_exists="replace", index=False)

def q(sql):
  return pd.read_sql(sql, conn)


In [ ]:
q("SELECT Neighborhood, COUNT(*) AS houses FROM houses GROUP BY Neighborhood ORDER BY houses DESC")

,Neighborhood,houses
0,NAmes,225
1,CollgCr,150
2,OldTown,113
3,Edwards,100
4,Somerst,86
5,Gilbert,79
6,NridgHt,77
7,Sawyer,74
8,NWAmes,73
9,SawyerW,59


In [ ]:
q("SELECT Neighborhood, ROUND(AVG(SalePrice)) AS avg_price FROM houses ORDER BY avg_price DESC ")

,Neighborhood,avg_price
0,CollgCr,180921.0


In [ ]:
q("""
SELECT Neighborhood,
  COUNT(*) AS houses,
  ROUND(AVG(SalePrice)) AS avg_price,
  MIN(SalePrice) AS cheapest,
  MAX(SalePrice) AS priciest
FROM houses
GROUP BY Neighborhood
ORDER BY avg_price DESC
""")

,Neighborhood,houses,avg_price,cheapest,priciest
0,NoRidge,41,335295.0,190000,755000
1,NridgHt,77,316271.0,154000,611657
2,StoneBr,25,310499.0,170000,556581
3,Timber,38,242247.0,137500,378500
4,Veenker,11,238773.0,162500,385000
5,Somerst,86,225380.0,144152,423000
6,ClearCr,28,212565.0,130000,328000
7,Crawfor,51,210625.0,90350,392500
8,CollgCr,150,197966.0,110000,424870
9,Blmngtn,17,194871.0,159895,264561


**HAVING: ONLY NEIGHBORHOODS WITH MORE THAN 50 HOUSES**

In [ ]:
q("""
SELECT Neighborhood, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY Neighborhood
HAVING COUNT(*) > 50
ORDER BY avg_price DESC
""")

,Neighborhood,houses,avg_price
0,NridgHt,77,316271.0
1,Somerst,86,225380.0
2,Crawfor,51,210625.0
3,CollgCr,150,197966.0
4,Gilbert,79,192855.0
5,NWAmes,73,189050.0
6,SawyerW,59,186556.0
7,NAmes,225,145847.0
8,Sawyer,74,136793.0
9,OldTown,113,128225.0


**GROUP BY QUALITY SCORE**

In [14]:
q("""
SELECT OverallQual, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY OverallQual
ORDER BY OverallQual
""")

,OverallQual,houses,avg_price
0,1,2,50150.0
1,2,3,51770.0
2,3,20,87474.0
3,4,116,108421.0
4,5,397,133523.0
5,6,374,161603.0
6,7,319,207716.0
7,8,168,274736.0
8,9,43,367513.0
9,10,18,438588.0


*Average price rises at every step of overall quality, from about 50,000 at quality 1 to 438,588 at quality 10. The extreme groups are small (2 and 18 houses), but the pattern is clear.*

**WHERE AND GROUP BY TOGETHER: HOUSES BUILT AFTER 2000**

In [ ]:
q("""
SELECT Neighborhood, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
WHERE YearBuilt > 2000
GROUP BY Neighborhood
ORDER BY avg_price DESC
""")

,Neighborhood,houses,avg_price
0,NoRidge,1,466500.0
1,StoneBr,14,381213.0
2,NridgHt,77,316271.0
3,Crawfor,3,300833.0
4,Timber,21,265036.0
5,Somerst,71,232922.0
6,NWAmes,1,227000.0
7,CollgCr,90,212947.0
8,Gilbert,34,204330.0
9,OldTown,1,200000.0


In [13]:
q("""
SELECT BedroomAbvGr, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY BedroomAbvGr
ORDER BY BedroomAbvGr
""")

,BedroomAbvGr,houses,avg_price
0,0,6,221493.0
1,1,50,173162.0
2,2,358,158198.0
3,3,804,181057.0
4,4,213,220421.0
5,5,21,180819.0
6,6,7,143779.0
7,8,1,200000.0


**JOIN: CHECK YOUR SESSION IS ALIVE**

In [16]:
q("SELECT COUNT(*) FROM houses")

,COUNT(*)
0,1460


**BUILD THE SUMMARY TABLE**

In [17]:
conn.execute("DROP TABLE IF EXISTS hood_stats")
conn.execute("""
CREATE TABLE hood_stats AS
SELECT Neighborhood, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY Neighborhood
""")
conn.commit()
q("SELECT * FROM hood_stats ORDER BY avg_price DESC LIMIT 5")

,Neighborhood,houses,avg_price
0,NoRidge,41,335295.0
1,NridgHt,77,316271.0
2,StoneBr,25,310499.0
3,Timber,38,242247.0
4,Veenker,11,238773.0


**JOIN THE TABLES**

In [18]:
q("""
SELECT h.neighborhood, h.SalePrice, s.avg_price,
       h.SalePrice - s.avg_price AS above_avg
FROM houses AS h
JOIN hood_stats AS s ON h.Neighborhood = s.Neighborhood
ORDER BY above_avg DESC
LIMIT 10
""")

,Neighborhood,SalePrice,avg_price,above_avg
0,NoRidge,755000,335295.0,419705.0
1,NoRidge,745000,335295.0,409705.0
2,OldTown,475000,128225.0,346775.0
3,NridgHt,611657,316271.0,295386.0
4,NoRidge,625000,335295.0,289705.0
5,NridgHt,582933,316271.0,266662.0
6,StoneBr,556581,310499.0,246082.0
7,NridgHt,555000,316271.0,238729.0
8,StoneBr,538000,310499.0,227501.0
9,CollgCr,424870,197966.0,226904.0


**COUNT THE HOUSES ABOVE THEIR NEIGHBORHOOD AVERAGE**

In [19]:
q("""
SELECT COUNT(*) AS above_their_hood_avg
FROM houses AS h
JOIN hood_stats AS s ON h.Neighborhood = s.Neighborhood
WHERE h.SalePrice > s.avg_price
""")

,above_their_hood_avg
0,636


*Joining each house to its neighborhood’s average shows which houses beat their local market. The biggest outperformer is a NoRidge house that sold 419,705 above its neighborhood average. 636 of the 1,460 houses (about 44%) sold above their neighborhood average, which is fewer than half because a few very expensive houses pull each average up. The OldTown house built in 1892 with quality 10 beat its neighborhood average by 346,775, so quality can outweigh location.*

**NOI AND CAP RATE**

In [20]:
price = 200000
monthly_rent = 1500
yearly_costs = 6000

noi = monthly_rent * 12 - yearly_costs
print("NOI:", noi)
print("Cap rate:", round(noi / price * 100, 1), "%")

NOI: 12000
Cap rate: 6.0 %
